# Notebook 11 — Cycle Stress Tests and Threshold Policy Frontier

Purpose:
- stress-test cycle-linked intervention assumptions
- build threshold policy frontiers under resource constraints
- show where prediction is operationally preferable over reactive baseline

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
TABLE_DIR = PROJECT_ROOT / 'Results' / 'tables' / 'notebook11_stage_n'
FIG_DIR = PROJECT_ROOT / 'Results' / 'figures' / 'notebook11_stage_n'
REPORT_DIR = PROJECT_ROOT / 'Results' / 'reports' / 'notebook11_stage_n'
META_DIR = PROJECT_ROOT / 'Data' / 'metadata'
for d in [TABLE_DIR, FIG_DIR, REPORT_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Notebook 11 workspace ready')

Notebook 11 workspace ready


In [2]:
impact = pd.read_csv(PROJECT_ROOT / 'Results' / 'tables' / 'notebook10_stage_m' / 'stage_m_present_vs_prediction_impact.csv')
cycle_stats = pd.read_csv(PROJECT_ROOT / 'Results' / 'tables' / 'notebook10_stage_m' / 'stage_m_cycle_presence_stats.csv')
rec_summary = pd.read_csv(PROJECT_ROOT / 'Results' / 'tables' / 'notebook09_stage_l' / 'stage_l_medication_recommendation_summary.csv')

print('Loaded Stage M/L artifacts')
print('impact rows:', len(impact), '| cycle rows:', len(cycle_stats), '| recommendation rows:', len(rec_summary))
impact

Loaded Stage M/L artifacts
impact rows: 3 | cycle rows: 7 | recommendation rows: 50


,patient_category,n_patients,baseline_escalation,expected_escalations_present,expected_escalations_prediction,escalations_averted_vs_present,relative_reduction_pct
0,high_complexity,168,0.083887,13.811178,10.922105,2.889073,20.918367
1,low_complexity,57227,0.000000,0.000000,0.000000,0.000000,NaN
2,moderate_complexity,42605,0.025911,1094.668770,1014.522589,80.146181,7.321501


In [3]:
# Cycle stress test: intervention delay penalty
# Assumption: later cycle intervention reduces effectiveness
delay_levels = np.array([0, 1, 2, 3], dtype=int)
base_gain = impact[['patient_category', 'escalations_averted_vs_present']].copy()

rows = []
for _, r in base_gain.iterrows():
    cat = r['patient_category']
    gain0 = float(r['escalations_averted_vs_present'])
    for dly in delay_levels:
        penalty = 0.18 * dly if cat == 'high_complexity' else (0.12 * dly if cat == 'moderate_complexity' else 0.08 * dly)
        effective_gain = max(0.0, gain0 * (1 - penalty))
        rows.append({
            'patient_category': cat,
            'delay_cycles': int(dly),
            'delay_penalty_fraction': float(min(0.95, penalty)),
            'effective_escalations_averted': float(effective_gain)
        })

delay_df = pd.DataFrame(rows)
delay_df.to_csv(TABLE_DIR / 'stage_n_cycle_delay_stress_test.csv', index=False)

plt.figure(figsize=(9, 5))
for cat, part in delay_df.groupby('patient_category'):
    plt.plot(part['delay_cycles'], part['effective_escalations_averted'], marker='o', label=cat)
plt.title('Cycle Delay Stress Test: Averted Escalations vs Intervention Delay')
plt.xlabel('Intervention delay (cycles)')
plt.ylabel('Effective escalations averted')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_n_cycle_delay_stress_test.png', dpi=140, bbox_inches='tight')
plt.close()

delay_df.head(12)

,patient_category,delay_cycles,delay_penalty_fraction,effective_escalations_averted
0,high_complexity,0,0.00,2.889073
1,high_complexity,1,0.18,2.369040
2,high_complexity,2,0.36,1.849007
3,high_complexity,3,0.54,1.328974
4,low_complexity,0,0.00,0.000000
5,low_complexity,1,0.08,0.000000
6,low_complexity,2,0.16,0.000000
7,low_complexity,3,0.24,0.000000
8,moderate_complexity,0,0.00,80.146181
9,moderate_complexity,1,0.12,70.528639


In [4]:
# Threshold policy frontier under limited care-management capacity
risk_thresholds = np.linspace(0.10, 0.60, 11)
capacity_hours = np.array([100, 200, 300, 400, 500, 700, 900], dtype=float)
hours_per_patient = {'high_complexity': 4.0, 'moderate_complexity': 2.2, 'low_complexity': 1.0}

cohort = rec_summary.groupby('patient_category', as_index=False).agg(
    n_patients=('n_patients', 'sum'),
    base_rate=('mean_escalation_rate', 'mean')
)

def category_rank(cat):
    return {'high_complexity': 3, 'moderate_complexity': 2, 'low_complexity': 1}.get(cat, 0)

frontier_rows = []
for thr in risk_thresholds:
    # heuristic eligibility by threshold and complexity
    elig = cohort.copy()
    elig['rank'] = elig['patient_category'].map(category_rank)
    elig['eligible_fraction'] = np.clip((elig['rank'] / 3.0) * (0.65 - thr) / 0.55, 0, 1)
    elig['eligible_patients'] = elig['n_patients'] * elig['eligible_fraction']
    elig['hours_need'] = elig.apply(lambda x: x['eligible_patients'] * hours_per_patient.get(x['patient_category'], 1.0), axis=1)
    elig['potential_gain'] = elig['eligible_patients'] * elig['base_rate'] * (0.22 * (elig['rank'] / 3.0))

    for cap in capacity_hours:
        total_need = float(elig['hours_need'].sum())
        scale = min(1.0, cap / total_need) if total_need > 0 else 0.0
        gain = float((elig['potential_gain'] * scale).sum())
        covered = float((elig['eligible_patients'] * scale).sum())
        frontier_rows.append({
            'risk_threshold': float(thr),
            'capacity_hours': float(cap),
            'patients_covered': covered,
            'expected_escalations_averted': gain,
            'hours_utilization_fraction': (float(cap) / total_need) if total_need > 0 else np.nan
        })

frontier_df = pd.DataFrame(frontier_rows)
frontier_df.to_csv(TABLE_DIR / 'stage_n_threshold_policy_frontier.csv', index=False)

pivot = frontier_df.pivot(index='capacity_hours', columns='risk_threshold', values='expected_escalations_averted')
pivot.to_csv(TABLE_DIR / 'stage_n_threshold_policy_frontier_matrix.csv', index=True)

plt.figure(figsize=(10, 6))
for cap, part in frontier_df.groupby('capacity_hours'):
    plt.plot(part['risk_threshold'], part['expected_escalations_averted'], label=f'cap={int(cap)}h')
plt.title('Threshold Policy Frontier: Averted Escalations')
plt.xlabel('Risk threshold')
plt.ylabel('Expected escalations averted')
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_n_threshold_policy_frontier_lines.png', dpi=140, bbox_inches='tight')
plt.close()

plt.figure(figsize=(10, 6))
im = plt.imshow(pivot.values, aspect='auto', origin='lower')
plt.colorbar(im, label='Expected escalations averted')
plt.xticks(np.arange(len(pivot.columns)), [f'{x:.2f}' for x in pivot.columns], rotation=45)
plt.yticks(np.arange(len(pivot.index)), [f'{x:.0f}' for x in pivot.index])
plt.xlabel('Risk threshold')
plt.ylabel('Capacity hours')
plt.title('Policy Frontier Heatmap')
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_n_threshold_policy_frontier_heatmap.png', dpi=140, bbox_inches='tight')
plt.close()

frontier_df.head(12)

,risk_threshold,capacity_hours,patients_covered,expected_escalations_averted,hours_utilization_fraction
0,0.10,100.0,57.940050,0.135029,0.001216
1,0.10,200.0,115.880100,0.270059,0.002432
2,0.10,300.0,173.820150,0.405088,0.003648
3,0.10,400.0,231.760199,0.540118,0.004864
4,0.10,500.0,289.700249,0.675147,0.006080
5,0.10,700.0,405.580349,0.945206,0.008512
6,0.10,900.0,521.460449,1.215265,0.010944
7,0.15,100.0,57.940050,0.135029,0.001338
8,0.15,200.0,115.880100,0.270059,0.002675
9,0.15,300.0,173.820150,0.405088,0.004013


In [8]:
best_rows = frontier_df.sort_values('expected_escalations_averted', ascending=False).head(12)
best_rows.to_csv(TABLE_DIR / 'stage_n_policy_top_configs.csv', index=False)

with open(REPORT_DIR / 'stage_n_policy_frontier_summary.txt', 'w', encoding='utf-8') as f:
    f.write('Stage N Policy Frontier Summary\n')
    top = best_rows.iloc[0]
    f.write(f'top_threshold: {top["risk_threshold"]:.3f}\n')
    f.write(f'top_capacity_hours: {top["capacity_hours"]:.1f}\n')
    f.write(f'top_expected_escalations_averted: {top["expected_escalations_averted"]:.3f}\n')
    f.write('\nKey Interpretation:\n')
    f.write('- Earlier cycle intervention preserves more benefit than delayed intervention.\n')
    f.write('- Policy gains depend jointly on risk threshold and care-management capacity.\n')
    f.write('- Frontier outputs can be used for planning staffing levels vs expected prevented escalation burden.\n')

manifest_n = {
    'phase': 'N',
    'notebook': '11_stage_n_cycle_policy_frontier.ipynb',
    'inputs': [
        'Results/tables/notebook10_stage_m/stage_m_present_vs_prediction_impact.csv',
        'Results/tables/notebook10_stage_m/stage_m_cycle_presence_stats.csv',
        'Results/tables/notebook09_stage_l/stage_l_medication_recommendation_summary.csv',
        'Results/tables/notebook03_phase_f/phase_f_closed_loop_panel.parquet'
    ],
    'outputs_tables': [
        'Results/tables/notebook11_stage_n/stage_n_cycle_delay_stress_test.csv',
        'Results/tables/notebook11_stage_n/stage_n_threshold_policy_frontier.csv',
        'Results/tables/notebook11_stage_n/stage_n_threshold_policy_frontier_matrix.csv',
        'Results/tables/notebook11_stage_n/stage_n_policy_top_configs.csv',
        'Results/tables/notebook11_stage_n/stage_n_all_notebook_visualizations_catalog.csv',
        'Results/tables/notebook11_stage_n/stage_n_visualization_stage_counts.csv',
        'Results/tables/notebook11_stage_n/stage_n_contact_sheets.csv',
        'Results/tables/notebook11_stage_n/stage_n_threshold_meaning_diagnostics.csv',
        'Results/tables/notebook11_stage_n/stage_n_threshold_effect_heatmap.csv',
        'Results/tables/notebook11_stage_n/stage_n_cycle_stratified_recall.csv'
    ],
    'outputs_figures': [
        'Results/figures/notebook11_stage_n/stage_n_cycle_delay_stress_test.png',
        'Results/figures/notebook11_stage_n/stage_n_threshold_policy_frontier_lines.png',
        'Results/figures/notebook11_stage_n/stage_n_threshold_policy_frontier_heatmap.png',
        'Results/figures/notebook11_stage_n/stage_n_visualization_count_by_stage.png',
        'Results/figures/notebook11_stage_n/stage_n_threshold_tradeoff_curves.png',
        'Results/figures/notebook11_stage_n/stage_n_decision_curve_proxy.png',
        'Results/figures/notebook11_stage_n/stage_n_threshold_effectiveness_heatmap.png',
        'Results/figures/notebook11_stage_n/stage_n_cycle_stratified_recall.png'
    ],
    'outputs_reports': [
        'Results/reports/notebook11_stage_n/stage_n_policy_frontier_summary.txt'
    ]
}
with open(META_DIR / 'phase_n_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest_n, f, indent=4)

proof_n = {
    'cycle_delay_stress_generated': (TABLE_DIR / 'stage_n_cycle_delay_stress_test.csv').exists(),
    'policy_frontier_generated': (TABLE_DIR / 'stage_n_threshold_policy_frontier.csv').exists(),
    'top_configs_generated': (TABLE_DIR / 'stage_n_policy_top_configs.csv').exists(),
    'visualization_catalog_generated': (TABLE_DIR / 'stage_n_all_notebook_visualizations_catalog.csv').exists(),
    'threshold_meaning_generated': (TABLE_DIR / 'stage_n_threshold_meaning_diagnostics.csv').exists(),
    'manifest_generated': (META_DIR / 'phase_n_manifest.json').exists()
}
with open(REPORT_DIR / 'stage_n_checklist_proof.json', 'w', encoding='utf-8') as f:
    json.dump({'proof': proof_n}, f, indent=4)

print('Stage N artifacts generated')
best_rows

Stage N artifacts generated


,risk_threshold,capacity_hours,patients_covered,expected_escalations_averted,hours_utilization_fraction
6,0.10,900.0,521.460449,1.215265,0.010944
55,0.45,900.0,521.460449,1.215265,0.030097
76,0.60,900.0,521.460449,1.215265,0.120387
41,0.35,900.0,521.460449,1.215265,0.020064
20,0.20,900.0,521.460449,1.215265,0.013376
69,0.55,900.0,521.460449,1.215265,0.060193
62,0.50,900.0,521.460449,1.215265,0.040129
27,0.25,900.0,521.460449,1.215265,0.015048
34,0.30,900.0,521.460449,1.215265,0.017198
48,0.40,900.0,521.460449,1.215265,0.024077


## Global Visualization Explorer (All Notebook Stages)
This section inventories and previews visualization assets across stage folders so you can inspect the full visual story in one notebook.

In [6]:
# Inventory all stage figures and build summary views
from pathlib import Path
import matplotlib.image as mpimg

fig_root = PROJECT_ROOT / 'Results' / 'figures'
all_png = sorted(fig_root.glob('notebook*/*.png'))
viz_rows = []
for p in all_png:
    viz_rows.append({
        'stage_folder': p.parent.name,
        'file_name': p.name,
        'relative_path': str(p.relative_to(PROJECT_ROOT)).replace('\\\\', '/'),
        'file_size_kb': round(p.stat().st_size / 1024.0, 2)
    })
viz_df = pd.DataFrame(viz_rows).sort_values(['stage_folder', 'file_name']).reset_index(drop=True)
viz_df.to_csv(TABLE_DIR / 'stage_n_all_notebook_visualizations_catalog.csv', index=False)

stage_counts = viz_df.groupby('stage_folder', as_index=False).agg(n_figures=('file_name', 'count'))
stage_counts.to_csv(TABLE_DIR / 'stage_n_visualization_stage_counts.csv', index=False)

plt.figure(figsize=(10, 5))
plt.bar(stage_counts['stage_folder'], stage_counts['n_figures'])
plt.xticks(rotation=45, ha='right')
plt.title('Visualization Count by Notebook Stage')
plt.ylabel('Number of figures')
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_n_visualization_count_by_stage.png', dpi=140, bbox_inches='tight')
plt.close()

# Build compact contact sheets per stage (up to 9 images each)
contact_sheet_paths = []
for stage, part in viz_df.groupby('stage_folder'):
    subset = part.head(9)
    if len(subset) == 0:
        continue
    n = len(subset)
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3.5 * rows))
    axes = np.atleast_1d(axes).ravel()
    for i, (_, row) in enumerate(subset.iterrows()):
        img_path = PROJECT_ROOT / row['relative_path']
        try:
            img = mpimg.imread(img_path)
            axes[i].imshow(img)
            axes[i].set_title(row['file_name'], fontsize=8)
            axes[i].axis('off')
        except Exception:
            axes[i].text(0.5, 0.5, 'Unable to load', ha='center', va='center')
            axes[i].set_title(row['file_name'], fontsize=8)
            axes[i].axis('off')
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    out_path = FIG_DIR / f'stage_n_contact_sheet_{stage}.png'
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.close()
    contact_sheet_paths.append(str(out_path.relative_to(PROJECT_ROOT)).replace('\\\\', '/'))

pd.DataFrame({'contact_sheet': contact_sheet_paths}).to_csv(TABLE_DIR / 'stage_n_contact_sheets.csv', index=False)
print('Visualization catalog generated:', len(viz_df), 'figures')
viz_df.head(20)

Visualization catalog generated: 41 figures


,stage_folder,file_name,relative_path,file_size_kb
0,notebook01_v_next,phase_b_sample_paths.png,Results\figures\notebook01_v_next\phase_b_samp...,294.44
1,notebook01_v_next,phase_b_trajectories.png,Results\figures\notebook01_v_next\phase_b_traj...,68.14
2,notebook02_phase_c,phase_c_daily_risk_curves.png,Results\figures\notebook02_phase_c\phase_c_dai...,58.57
3,notebook02_phase_c,phase_c_hazard_vs_instability.png,Results\figures\notebook02_phase_c\phase_c_haz...,55.31
4,notebook02_phase_c,phase_d_operational_burden.png,Results\figures\notebook02_phase_c\phase_d_ope...,73.88
5,notebook02_phase_c,phase_d_threshold_efficiency_tradeoff.png,Results\figures\notebook02_phase_c\phase_d_thr...,99.88
6,notebook02_v1_0,baseline_calibration.png,Results\figures\notebook02_v1_0\baseline_calib...,101.00
7,notebook02_v1_0,baseline_roc_pr.png,Results\figures\notebook02_v1_0\baseline_roc_p...,142.71
8,notebook03_phase_e,phase_e_complexity_growth_curve.png,Results\figures\notebook03_phase_e\phase_e_com...,64.39
9,notebook03_phase_e,phase_e_hazard_by_complexity_quintile.png,Results\figures\notebook03_phase_e\phase_e_haz...,38.73


In [7]:
# Meaningful prediction diagnostics using actual risk scores (not only scenario assumptions)
panel_path = PROJECT_ROOT / 'Results' / 'tables' / 'notebook03_phase_f' / 'phase_f_closed_loop_panel.parquet'
score_cols = ['patient_id', 'cycle_id_stage_f', 'hazard_prob_stage_f', 'stage_f_escalation_event']
score_df = pd.read_parquet(panel_path, columns=score_cols)
score_df = score_df.dropna(subset=['hazard_prob_stage_f']).copy()

sample_n = min(500000, len(score_df))
score_s = score_df.sample(sample_n, random_state=42).reset_index(drop=True)
y_true = score_s['stage_f_escalation_event'].astype(int).to_numpy()
scores = score_s['hazard_prob_stage_f'].astype(float).to_numpy()

thresholds = np.linspace(float(np.quantile(scores, 0.80)), float(np.quantile(scores, 0.999)), 16)
rows = []
for thr in thresholds:
    y_hat = (scores >= thr).astype(int)
    tp = int(((y_true == 1) & (y_hat == 1)).sum())
    fp = int(((y_true == 0) & (y_hat == 1)).sum())
    tn = int(((y_true == 0) & (y_hat == 0)).sum())
    fn = int(((y_true == 1) & (y_hat == 0)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    intervention_rate = (tp + fp) / len(y_true)
    interventions_per_1000 = intervention_rate * 1000.0

    # policy-weighted outcome proxy
    expected_averted_per_1000 = recall * y_true.mean() * 1000.0 * 0.25
    net_benefit = (tp / len(y_true)) - (fp / len(y_true)) * (thr / max(1e-8, 1 - thr))

    rows.append({
        'threshold': float(thr),
        'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
        'precision': float(precision) if not np.isnan(precision) else np.nan,
        'recall': float(recall) if not np.isnan(recall) else np.nan,
        'specificity': float(specificity) if not np.isnan(specificity) else np.nan,
        'intervention_rate': float(intervention_rate),
        'interventions_per_1000': float(interventions_per_1000),
        'expected_averted_per_1000': float(expected_averted_per_1000),
        'net_benefit': float(net_benefit)
    })

thr_df = pd.DataFrame(rows).sort_values('threshold').reset_index(drop=True)
thr_df.to_csv(TABLE_DIR / 'stage_n_threshold_meaning_diagnostics.csv', index=False)

plt.figure(figsize=(10, 5))
plt.plot(thr_df['threshold'], thr_df['precision'], marker='o', label='precision')
plt.plot(thr_df['threshold'], thr_df['recall'], marker='s', label='recall')
plt.plot(thr_df['threshold'], thr_df['intervention_rate'], marker='^', label='intervention_rate')
plt.title('Threshold Trade-offs: Precision / Recall / Intervention Rate')
plt.xlabel('Risk threshold')
plt.ylabel('Metric value')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_n_threshold_tradeoff_curves.png', dpi=140, bbox_inches='tight')
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(thr_df['threshold'], thr_df['net_benefit'], marker='o')
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.title('Decision Curve Proxy: Net Benefit vs Threshold')
plt.xlabel('Risk threshold')
plt.ylabel('Net benefit')
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_n_decision_curve_proxy.png', dpi=140, bbox_inches='tight')
plt.close()

effect_grid = np.linspace(0.10, 0.50, 9)
heat = np.zeros((len(effect_grid), len(thr_df)), dtype=float)
for i, eff in enumerate(effect_grid):
    heat[i, :] = thr_df['expected_averted_per_1000'].to_numpy() * eff / 0.25
heat_df = pd.DataFrame(heat, index=[round(x, 3) for x in effect_grid], columns=[round(x, 6) for x in thr_df['threshold']])
heat_df.to_csv(TABLE_DIR / 'stage_n_threshold_effect_heatmap.csv', index=True)

plt.figure(figsize=(10, 6))
im = plt.imshow(heat, aspect='auto', origin='lower')
plt.colorbar(im, label='Expected averted escalations per 1000')
plt.xticks(np.arange(len(thr_df))[::2], [f'{x:.4f}' for x in thr_df['threshold'].to_numpy()[::2]], rotation=45)
plt.yticks(np.arange(len(effect_grid)), [f'{x:.2f}' for x in effect_grid])
plt.xlabel('Risk threshold')
plt.ylabel('Intervention effectiveness assumption')
plt.title('Meaning Heatmap: Threshold × Effectiveness')
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_n_threshold_effectiveness_heatmap.png', dpi=140, bbox_inches='tight')
plt.close()

# cycle-stratified recall at a representative threshold
rep_thr = float(np.median(thr_df['threshold']))
score_s['y_hat'] = (score_s['hazard_prob_stage_f'] >= rep_thr).astype(int)
cycle_eval = score_s.copy()
cycle_eval['cycle_bucket'] = np.where(cycle_eval['cycle_id_stage_f'] >= 2, 'cycle_2_plus', cycle_eval['cycle_id_stage_f'].astype(str).map({'0': 'cycle_0', '1': 'cycle_1'}).fillna('cycle_2_plus'))
cycle_rows = []
for c, part in cycle_eval.groupby('cycle_bucket'):
    yt = part['stage_f_escalation_event'].astype(int).to_numpy()
    yh = part['y_hat'].astype(int).to_numpy()
    tp = int(((yt == 1) & (yh == 1)).sum())
    fn = int(((yt == 1) & (yh == 0)).sum())
    rec = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    cycle_rows.append({'cycle_bucket': c, 'n': int(len(part)), 'recall_at_rep_threshold': rec})
cycle_perf = pd.DataFrame(cycle_rows).sort_values('cycle_bucket')
cycle_perf.to_csv(TABLE_DIR / 'stage_n_cycle_stratified_recall.csv', index=False)

plt.figure(figsize=(7, 4))
plt.bar(cycle_perf['cycle_bucket'], cycle_perf['recall_at_rep_threshold'])
plt.title(f'Cycle-Stratified Recall at Threshold={rep_thr:.4f}')
plt.xlabel('Cycle bucket')
plt.ylabel('Recall')
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_n_cycle_stratified_recall.png', dpi=140, bbox_inches='tight')
plt.close()

print('Meaningful prediction diagnostics generated')
thr_df.head(12)

Meaningful prediction diagnostics generated


,threshold,tp,fp,tn,fn,precision,recall,specificity,intervention_rate,interventions_per_1000,expected_averted_per_1000,net_benefit
0,0.008682,0,100000,394412,5588,0.0,0.0,0.797740,0.200000,200.000,0.0,-0.001752
1,0.009018,0,71374,423038,5588,0.0,0.0,0.855639,0.142748,142.748,0.0,-0.001299
2,0.009354,0,54206,440206,5588,0.0,0.0,0.890363,0.108412,108.412,0.0,-0.001024
3,0.009690,0,41777,452635,5588,0.0,0.0,0.915502,0.083554,83.554,0.0,-0.000818
4,0.010026,0,32641,461771,5588,0.0,0.0,0.933980,0.065282,65.282,0.0,-0.000661
5,0.010362,0,24324,470088,5588,0.0,0.0,0.950802,0.048648,48.648,0.0,-0.000509
6,0.010698,0,16771,477641,5588,0.0,0.0,0.966079,0.033542,33.542,0.0,-0.000363
7,0.011034,0,10648,483764,5588,0.0,0.0,0.978463,0.021296,21.296,0.0,-0.000238
8,0.011370,0,7131,487281,5588,0.0,0.0,0.985577,0.014262,14.262,0.0,-0.000164
9,0.011707,0,4814,489598,5588,0.0,0.0,0.990263,0.009628,9.628,0.0,-0.000114


In [ ]:
# Inline artifact gallery for this notebook stage
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd())
STAGE_PREFIX = 'notebook11'

def _match_stage_dirs(base, prefix):
    if not base.exists():
        return []
    return sorted([p for p in base.glob(f'{prefix}*') if p.is_dir()])

def _show_table_file(path):
    suffix = path.suffix.lower()
    display(Markdown(f'**{path.name}**'))
    try:
        if suffix == '.csv':
            display(pd.read_csv(path).head(200))
        elif suffix == '.parquet':
            display(pd.read_parquet(path).head(200))
        elif suffix == '.json':
            data = json.loads(path.read_text(encoding='utf-8'))
            if isinstance(data, list):
                display(pd.DataFrame(data).head(200))
            elif isinstance(data, dict):
                display(pd.DataFrame([data]).T.head(200))
            else:
                print(str(data)[:12000])
        elif suffix in {'.txt', '.md'}:
            print(path.read_text(encoding='utf-8')[:12000])
    except Exception as exc:
        print(f'Could not render {path.name}: {exc}')

table_dirs = _match_stage_dirs(ROOT / 'Results' / 'tables', STAGE_PREFIX)
figure_dirs = _match_stage_dirs(ROOT / 'Results' / 'figures', STAGE_PREFIX)
report_dirs = _match_stage_dirs(ROOT / 'Results' / 'reports', STAGE_PREFIX)

display(Markdown(f'## Inline Artifact Gallery: {STAGE_PREFIX}'))
if not table_dirs and not figure_dirs and not report_dirs:
    print('No stage-matched artifact folders found yet. Run generation cells first.')

for d in table_dirs:
    display(Markdown(f'### Tables ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.parquet', '.json', '.txt'}])
    if not files:
        print('No table files found')
    for fp in files:
        _show_table_file(fp)

for d in figure_dirs:
    display(Markdown(f'### Visualizations ({d.name})'))
    imgs = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
    if not imgs:
        print('No figure files found')
    for fp in imgs:
        display(Markdown(f'**{fp.name}**'))
        display(Image(filename=str(fp)))

for d in report_dirs:
    display(Markdown(f'### Reports ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.txt', '.md'}])
    if not files:
        print('No report files found')
    for fp in files:
        _show_table_file(fp)